<a href="https://colab.research.google.com/github/p-perrone/UiO_AdvancedRemoteSensing/blob/notekooks/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import packages
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import altair as alt
import random
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display
from scipy.stats import linregress
cw = 8.325/2.54
tw = 17/2.54
lw = 12/2.54
sns.set()
sns.set_style("darkgrid", {"axes.facecolor": ".95"})
from matplotlib import rcParams
rcParams.update({
    "font.size": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    'axes.titlesize':9,
    'legend.fontsize': 8,
})


In [2]:
from google.colab import files
def saveplot(plotname):
    """Salva e scarica il plot corrente"""
    filename = f"{plotname}.pdf"
    plt.savefig(filename)
    files.download(filename)

from pathlib import Path

cw = 8.325/2.54
tw = 17/2.54

In [3]:
# Run authentication
ee.Authenticate()
ee.Initialize(project='dulcet-iterator-470310-n0') # PUT YOUR API KEY (Project ID) HERE!


In [41]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [10]:
# @title
drawn_features = Map.draw_last_feature
drawn_rectangle = drawn_features

if drawn_rectangle:
    # Get the geometry of the feature
    rectangle_geometry = drawn_rectangle.geometry()

    # Get the coordinates of the geometry as a Python object
    coordinates = rectangle_geometry.getInfo()['coordinates']

    # Print the coordinates
    print("Coordinates of the drawn rectangle:")
    print(coordinates)
else:
    print("Cannot extract coordinates as no rectangle feature is available.")

savegeom = ee.Geometry.Polygon(coordinates)
savegeom

Coordinates of the drawn rectangle:
[[[-65.419631, 66.419701], [-65.365722, 66.411803], [-65.360571, 66.416439], [-65.416541, 66.423271], [-65.419631, 66.419701]]]


ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -65.419631,
              66.419701
            ],
            [
              -65.365722,
              66.411803
            ],
            [
              -65.360571,
              66.416439
            ],
            [
              -65.416541,
              66.423271
            ],
            [
              -65.419631,
              66.419701
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})

In [5]:
pangnirtung_geom = ee.Geometry.Polygon(
  [[[-65.722299, 66.12121],
    [-65.722299, 66.141076],
    [-65.642668, 66.141076],
    [-65.642668, 66.12121],
    [-65.722299, 66.12121]]],
  proj='EPSG:4326'

)


pangnirtung_point = ee.Geometry.Point([65.71, 66.14], proj='EPSG:4326')

In [44]:
kolik_valley = ee.Geometry.Polygon(
    [[[-65.84997, 66.273143], [-65.825587, 66.260983], [-65.806526, 66.266649], [-65.830738, 66.278806], [-65.84997, 66.273143]]]
)
ungujalik_cape = ee.Geometry.Polygon(
    [[[-65.922052, 65.9764], [-65.922052, 65.992329], [-65.87775, 65.992329], [-65.87775, 65.9764], [-65.922052, 65.9764]]]
)
ujarasujjualuk_plateau = ee.Geometry.Polygon(
    [[[-65.740622, 66.106752], [-65.740622, 66.122322],  [-65.698723, 66.122322], [-65.698723, 66.106752], [-65.740622, 66.106752]]]
)
aulattiviup_fiordshore = ee.Geometry.Polygon(
    [[[-65.502958, 66.318069], [-65.502958, 66.327582], [-65.476085, 66.327582], [-65.476085, 66.318069], [-65.502958, 66.318069]]]
)
tumbling_glacier = ee.Geometry.Polygon(
    [[[-65.623876, 66.456344], [-65.623876, 66.465188], [-65.593311, 66.465188], [-65.593311, 66.456344], [-65.623876, 66.456344]]]
)
gauntlet_northface = ee.Geometry.Polygon(
    [[[-65.419631, 66.419701], [-65.365722, 66.411803], [-65.360571, 66.416439], [-65.416541, 66.423271], [-65.419631, 66.419701]]]
)

geoms = [kolik_valley, ungujalik_cape, ujarasujjualuk_plateau, aulattiviup_fiordshore, tumbling_glacier, gauntlet_northface]

In [45]:
import ee.batch

# Create a list to hold ee.Feature objects
features_to_export = []

# Iterate through geometries and their names to create features
for i, geom in enumerate(geoms):
    feature = ee.Feature(geom, {'name': geoms[i]})
    features_to_export.append(feature)

# Create an ee.FeatureCollection from the list of features
feature_collection_to_export = ee.FeatureCollection(features_to_export)

# Define export parameters
task = ee.batch.Export.table.toDrive(**{
    'collection': feature_collection_to_export,
    'description': 'Exported_Geometries_SHP',
    'folder': 'ee_exports',
    'fileNamePrefix': 'all_geometries',
    'fileFormat': 'SHP'
})

# Start the export task
task.start()

print(f"Export task '{task.status()['description']}' started. Check your Google Drive for 'all_geometries.zip' in the 'ee_exports' folder.")

Export task 'Exported_Geometries_SHP' started. Check your Google Drive for 'all_geometries.zip' in the 'ee_exports' folder.


In [55]:
modis = ee.ImageCollection('MODIS/061/MOD11A1').filterDate('2002-01-01', '2025-01-01')
Map.addLayer(modis.select('LST_Day_1km'), {'min': 7000, 'max': 16500, 'palette': 'turbo'}, 'LST_Day_1km', False, 0.98)
Map.centerObject(pangnirtung_geom, zoom=8)
Map.add_basemap("SATELLITE")
Map

Map(bottom=728.0, center=[66.13114682190704, -65.68248350000003], controls=(WidgetControl(options=['position',…

In [56]:
Map.addLayer(kolik_valley, {'color': 'purple'}, 'Kolik Valley');
Map.addLayer(ungujalik_cape, {'color': 'orange'}, 'Ungujalik Cape');
Map.addLayer(ujarasujjualuk_plateau, {'color': 'cyan'}, 'Ujarasujjualuk Plateau');
Map.addLayer(aulattiviup_fiordshore, {'color': 'magenta'}, 'Aulattiviup Fiordshore');
Map.addLayer(tumbling_glacier, {'color': 'yellow'}, 'Tumbling Glacier');

# gauntlet_northface is defined as a list of coordinates, not an ee.Geometry.Polygon, so it cannot be added directly.
Map

Map(bottom=728.0, center=[66.13114682190704, -65.68248350000003], controls=(WidgetControl(options=['position',…

In [57]:
Map.addLayer(gauntlet_northface, {'color': 'red'}, 'Gauntlet Northface');
Map

Map(bottom=16857.0, center=[66.13114682190704, -65.68248350000003], controls=(WidgetControl(options=['position…

In [19]:
# GEMINI HELPED
def add_date_properties(image):
    # Get the timestamp of the image
    date = ee.Date(image.get('system:time_start'))

    # Extract year and month
    year = date.get('year')
    month = date.get('month')

    # Add year and month as properties to the image
    return image.set({'year': year, 'month': month})


In [27]:
# GEMINI HELPED
monthly_lst_data = []
means = list()
stds = list()
q25s = list()
q75s = list()

start_year = 2002
end_year = 2025
modis_factor = 0.02
MODIS_SCALE = 1000

for geom in geoms:
  for year in range(start_year, end_year + 1):
      print(f"Processing year: {year}")
      for month in range(1, 13):
          start_date = ee.Date.fromYMD(year, month, 1)
          end_date = start_date.advance(1, 'month')

          monthly_collection = modis.filterDate(start_date, end_date).filterBounds(pangnirtung_geom)

          # Select the 'LST_Day_1km' band
          monthly_lst = monthly_collection.select('LST_Day_1km')

          # Check if the collection is empty before reducing
          if monthly_lst.size().getInfo() == 0:
              mean_lst = None
          else:
              # Compute the mean of the LST band for the month
              # First, reduce the collection to a single image using the mean reducer
              mean_val = (
                  monthly_lst.mean()                      # Image
                  .reduceRegion(
                      reducer = ee.Reducer.mean(),
                      geometry = geom,
                      scale = MODIS_SCALE,
                      maxPixels = 1e13
                  )
                  .get('LST_Day_1km')
                  .getInfo()
              )

              std_val = (
                  monthly_lst.reduce(ee.Reducer.stdDev())
                  .reduceRegion(
                      reducer = ee.Reducer.mean(),
                      geometry = geom,
                      scale = MODIS_SCALE,
                      maxPixels = 1e13
                  )
                  .get('LST_Day_1km_stdDev')
                  .getInfo()
              )

              q25_val = (
                  monthly_lst.reduce(ee.Reducer.percentile([25]))
                  .reduceRegion(
                      reducer = ee.Reducer.mean(),
                      geometry = geom,
                      scale = MODIS_SCALE,
                      maxPixels = 1e13
                  )
                  .get('LST_Day_1km_p25')
                  .getInfo()
              )

              q75_val = (
                  monthly_lst.reduce(ee.Reducer.percentile([75]))
                  .reduceRegion(
                      reducer = ee.Reducer.mean(),
                      geometry = geom,
                      scale = MODIS_SCALE,
                      maxPixels = 1e13
                  )
                  .get('LST_Day_1km_p75')
                  .getInfo()
              )

              means.append(mean_val*modis_factor - 273.15)
              stds.append(std_val*modis_factor)
              q25s.append(q25_val*modis_factor - 273.15)
              q75s.append(q75_val*modis_factor - 273.15)

Processing year: 2002


KeyboardInterrupt: 

In [21]:
years = np.arange(1, 277, 1)

plt.figure(figsize=(10,4))
plt.plot(years, means)
plt.plot(years, np.array(means)+np.array(stds)/270)
plt.plot(years, np.array(means)-np.array(stds)/270)


NameError: name 'means' is not defined

<Figure size 1000x400 with 0 Axes>

In [22]:
years = []
months = []

for i in range(len(means)):
    year = start_year + (i // 12)
    month = (i % 12) + 1
    years.append(year)
    months.append(month)

df = pd.DataFrame({
    "year": years,
    "month": months,
    "mean": means,
    "std" : stds,
    "q25" : q25s,
    "q75" : q75s

})

print(df)



NameError: name 'means' is not defined

In [23]:
df_years = df.groupby('year').mean()
df_month = df.groupby('month').mean()

years = np.arange(2002, 2025, 1)
months = np.arange(1, 13, 1)

colors = sns.color_palette('deep')

# fitting the mean curve
slope, intercept, r, p, se = linregress(years, df_years['mean'])
x = np.linspace(2002, 2025, 100)
y = slope * x + intercept

fig, axes = plt.subplots(2, 1, figsize=(lw, 5))
ax0 = axes[0]
ax1 = axes[1]

# ------ annual means ----------
ax0.plot(df_years["mean"], c=colors[0])
ax0.fill_between(df_years.index, df_years["q25"], df_years["q75"],
                 alpha=0.3,
                 label='Interquantile 25% - 75%',
                 color=colors[0],
                 edgecolor='none')
ax0.set_ylabel("Temperature (°C)")
ax0.set_xlabel("Year")
ax0.set_title("Mean Temperature (°C) per Year")

ax0.plot(x, y, label=f'Linear fit: $y = ${slope:.2f}$x ${intercept:.2f}',
         c=colors[3])
ax0.legend()

# ------ monthly means ----------
ax1.plot(df_month["mean"], c=colors[1])
ax1.set_title("Mean Temperature (°C) per Month")
ax1.fill_between(df_month.index, df_month["q25"], df_month["q75"],
                 alpha=0.3,
                 label='Interquantile 25% - 75%',
                 color=colors[1],
                 edgecolor='none')
ax1.legend()
ax1.set_ylabel("Temperature (°C)")
ax1.set_xlabel("Month")
ax1.set_xticks(months)

plt.tight_layout()
saveplot('modis_year+month')

NameError: name 'df' is not defined

In [24]:
from matplotlib import rcParams
from google.colab import files
rcParams.update({
    # Font sizes (per assomigliare a LaTeX article class)
    "font.size": 9,           # ~ \normalsize
    "axes.labelsize": 9,      # Labels per assi
    "xtick.labelsize": 8,      # Numeri sugli assi - più piccoli
    "ytick.labelsize": 8,
    "legend.fontsize": 8,      # Legenda
    "figure.titlesize": 10,    # Titolo sopra la figura
    "figure.labelsize": 10,
})

def saveplot(plotname):
    """Salva e scarica il plot corrente"""
    filename = f"{plotname}.pdf"
    plt.savefig(filename)
    files.download(filename)

from pathlib import Path

cw = 8.325/2.54
tw = 17/2.54

In [28]:
start_year = 2002
end_year = 2025
modis_factor = 0.02
MODIS_SCALE = 1000

terra = (ee.ImageCollection("MODIS/061/MOD11A1")
         .select("LST_Day_1km")
         .filterDate(f"{start_year}-01-01", f"{end_year}-12-31"))

aqua = (ee.ImageCollection("MODIS/061/MYD11A1")
         .select("LST_Day_1km")
         .filterDate(f"{start_year}-01-01", f"{end_year}-12-31"))

terra_fc = terra.map(
    lambda img:
        img.sample(region=pangnirtung_point, scale=MODIS_SCALE)
           .first()
           .set("date", img.date().format("YYYY-MM-dd"))
)
aqua_fc = aqua.map(
    lambda img:
        img.sample(region=pangnirtung_point, scale=MODIS_SCALE)
           .first()
           .set("date", img.date().format("YYYY-MM-dd"))
)

def collection_to_df(collection):
    # Get list of images
    imgs = collection.toList(collection.size())

    records = []
    for i in range(collection.size().getInfo()):
        img = ee.Image(imgs.get(i))
        sample = img.sample(region=pangnirtung_point, scale=MODIS_SCALE).first()
        if sample:  # skip nulls
            info = sample.getInfo()
            if 'properties' in info:
                records.append(info['properties'])

    df = pd.DataFrame(records)
    # Convert LST_Day_1km to numeric and drop rows with NaNs
    df['LST_Day_1km'] = pd.to_numeric(df['LST_Day_1km'], errors='coerce')
    df = df.dropna(subset=['LST_Day_1km'])
    return df

# --- Convert collections to DataFrames ---
terra_df = collection_to_df(terra_fc)
aqua_df = collection_to_df(aqua_fc)

# --- Merge Terra and Aqua by date ---
combined_df = pd.merge(terra_df, aqua_df, on='date', suffixes=('_terra', '_aqua'))
combined_df




EEException: Collection.toList: Error in map(ID=2002_01_01):
Element.set: Parameter 'object' is required and may not be null.

# Task
Filter the MODIS/061/MOD11A1 and MODIS/061/MYD11A1 image collections for the year 2003, create mosaics from the filtered collections, and add these mosaics as layers titled 'Terra LST Day 2003' and 'Aqua LST Day 2003' respectively to the Earth Engine map, then display the updated map.

## Filter Terra Collection

### Subtask:
Filter the MODIS/061/MOD11A1 image collection to include only images from the year 2003.


**Reasoning**:
First, I will define the `terra` image collection as specified in the instructions. Then, I will filter this collection to include only images from the year 2003 and store the result in `terra_2003`.



In [29]:
terra = ee.ImageCollection('MODIS/061/MOD11A1')
terra_2003 = terra.filterDate('2003-01-01', '2003-12-31')
print(f"Number of images in terra_2003 collection: {terra_2003.size().getInfo()}")

Number of images in terra_2003 collection: 357


## Filter Aqua Collection

### Subtask:
Filter the MODIS/061/MYD11A1 image collection to include only images from the year 2003.

**Reasoning**:
Following the previous step, I will now define the `aqua` image collection and filter it to include only images from the year 2003, storing the result in `aqua_2003`.



In [30]:
aqua = ee.ImageCollection('MODIS/061/MYD11A1')
aqua_2003 = aqua.filterDate('2003-01-01', '2003-12-31')
print(f"Number of images in aqua_2003 collection: {aqua_2003.size().getInfo()}")

Number of images in aqua_2003 collection: 364


## Create Terra Mosaic

### Subtask:
Create a mosaic from the filtered `terra_2003` image collection.

**Reasoning**:
To create a mosaic from the `terra_2003` image collection, I will apply the `.mosaic()` method to combine all images into a single one, and then select the 'LST_Day_1km' band as it is the relevant band for the task.



In [31]:
terra_mosaic = terra_2003.mosaic().select('LST_Day_1km')
print(f"Terra mosaic created: {terra_mosaic.getInfo()}")

Terra mosaic created: {'type': 'Image', 'bands': [{'id': 'LST_Day_1km', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]}


## Create Aqua Mosaic

### Subtask:
Create a mosaic from the filtered `aqua_2003` image collection.

**Reasoning**:
To create a mosaic from the `aqua_2003` image collection, I will apply the `.mosaic()` method to combine all images into a single one, and then select the 'LST_Day_1km' band, similar to the Terra mosaic.



In [32]:
aqua_mosaic = aqua_2003.mosaic().select('LST_Day_1km')
print(f"Aqua mosaic created: {aqua_mosaic.getInfo()}")

Aqua mosaic created: {'type': 'Image', 'bands': [{'id': 'LST_Day_1km', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]}


## Add Mosaics to Map

### Subtask:
Add the `terra_mosaic` and `aqua_mosaic` as layers to the Earth Engine map, titled 'Terra LST Day 2003' and 'Aqua LST Day 2003' respectively.

**Reasoning**:
With both the Terra and Aqua mosaics successfully created, I will now add them as layers to the interactive Earth Engine map using `Map.addLayer()`, ensuring they are visible with appropriate visualization parameters and distinct titles, as specified in the subtask.



In [33]:
vis_params = {'min': 7000, 'max': 16500, 'palette': 'turbo'}

Map.addLayer(terra_mosaic, vis_params, 'Terra LST Day 2003')
Map.addLayer(aqua_mosaic, vis_params, 'Aqua LST Day 2003')

Map

Map(bottom=1051993.0, center=[66.40640962300085, -65.43551368607504], controls=(WidgetControl(options=['positi…

## Final Task

### Subtask:
Confirm that the Terra and Aqua mosaics for 2003 have been loaded and displayed on the map, allowing for visual inspection of the data.


## Summary:

### Q&A
Yes, the Terra and Aqua mosaics for 2003 were successfully loaded and displayed on the Earth Engine map, allowing for visual inspection.

### Data Analysis Key Findings
*   The `MODIS/061/MOD11A1` (Terra) image collection for 2003 contained 357 images.
*   The `MODIS/061/MYD11A1` (Aqua) image collection for 2003 contained 364 images.
*   Mosaics for both Terra and Aqua collections for 2003 were successfully created by combining all images within each collection and selecting the 'LST\_Day\_1km' band.
*   Both the 'Terra LST Day 2003' and 'Aqua LST Day 2003' mosaics were successfully added to the Earth Engine map with a 'turbo' palette and a visualization range from 7000 to 16500.

### Insights or Next Steps
*   The displayed mosaics enable a direct visual comparison of Land Surface Temperature (LST) between Terra and Aqua satellites for the year 2003.
*   Further analysis could involve calculating the difference or correlation between the Terra and Aqua LST mosaics to assess their agreement or discrepancies for the specified year.


# Task
Collect all defined `ee.Geometry` objects, including `pangnirtung_geom`, `savegeom`, and `gauntlet_northface_geom`, and consolidate them into a single list named `all_geometries` for further processing.

## Prepare Geometries

### Subtask:
Collect all defined `ee.Geometry` objects into a list for iteration.


**Reasoning**:
I will create a new list called `all_geometries` and append all the specified `ee.Geometry` objects to it. Then, I will print the list to confirm its contents.



In [72]:
geoms = [
    aulattiviup_fiordshore,
    ungujalik_cape,
    kolik_valley,
    ujarasujjualuk_plateau,
    gauntlet_northface,
    tumbling_glacier
]

geom_names = [
    "aulattiviup_fiordshore",
    "ungujalik_cape",
    "kolik_valley",
    "ujarasujjualuk_plateau",
    "gauntlet_northface",
    "tumbling_glacier"
]

print(f"All collected geometries: {geoms}")

All collected geometries: [ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -65.502958,
              66.318069
            ],
            [
              -65.502958,
              66.327582
            ],
            [
              -65.476085,
              66.327582
            ],
            [
              -65.476085,
              66.318069
            ],
            [
              -65.502958,
              66.318069
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
}), ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -65.922052,
              65.9764
            ],
            [
              -65.922052,
    

**Reasoning**:
The previous code failed because `savegeom` was not defined in the current execution context. I will explicitly redefine `savegeom` using the coordinates extracted from the previous execution of the map drawing feature, and then construct the `all_geometries` list.



In [47]:
savegeom = ee.Geometry.Polygon(
    [[[-65.623876, 66.456344], [-65.623876, 66.465188], [-65.593311, 66.465188], [-65.593311, 66.456344], [-65.623876, 66.456344]]]
)

all_geometries = [
    # pangnirtung_geom,
    # savegeom,
    # kolik_valley,
    # ungujalik_cape,
    # ujarasujjualuk_plateau,
    # aulattiviup_fiordshore,
    # tumbling_glacier,
    gauntlet_northface
]

print(f"All collected geometries: {all_geometries}")

All collected geometries: [ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Polygon",
    "arguments": {
      "coordinates": {
        "constantValue": [
          [
            [
              -65.419631,
              66.419701
            ],
            [
              -65.365722,
              66.411803
            ],
            [
              -65.360571,
              66.416439
            ],
            [
              -65.416541,
              66.423271
            ],
            [
              -65.419631,
              66.419701
            ]
          ]
        ]
      },
      "evenOdd": {
        "constantValue": true
      }
    }
  }
})]


## Process Each Geometry and Year

### Subtask:
For each geometry and for each year (2002-2025), filter the Terra (`MOD11A1`) and Aqua (`MYD11A1`) collections, combine them, and calculate the mean, standard deviation, median, 25th percentile, and 75th percentile of the 'LST_Day_1km' band. Convert LST values to Celsius.


**Reasoning**:
First, I need to define the image collections for Terra and Aqua, the `geometry_names` list to associate with `all_geometries`, and the `MODIS_SCALE` and `modis_factor` constants. Then, I will iterate through each geometry and year, filter and combine the collections, calculate the required statistics, convert them to Celsius, and store the results in `all_yearly_stats`.



In [66]:
terra_collection = ee.ImageCollection('MODIS/061/MOD11A1')
aqua_collection = ee.ImageCollection('MODIS/061/MYD11A1')

geometry_names = [
    'gauntlet_northface'
]

all_yearly_stats = []

start_year = 2002
end_year = 2025
MODIS_SCALE = 1000
modis_factor = 0.02

for i, geom in enumerate(all_geometries):
    geometry_name = geometry_names[i]
    print(f"\nProcessing geometry: {geometry_name}")
    for year in range(start_year, end_year + 1):
        # print(f"  Processing year: {year}")
        start_date = ee.Date.fromYMD(year, 1, 1)
        end_date = ee.Date.fromYMD(year, 12, 31)

        # Filter Terra and Aqua collections for the current year and geometry
        terra_yearly_collection = terra_collection.filterDate(start_date, end_date).filterBounds(geom).select('LST_Day_1km')
        aqua_yearly_collection = aqua_collection.filterDate(start_date, end_date).filterBounds(geom).select('LST_Day_1km')

        # Merge Terra and Aqua collections
        combined_yearly_collection = terra_yearly_collection.merge(aqua_yearly_collection)

        # Check if the combined collection is empty
        if combined_yearly_collection.size().getInfo() == 0:
            yearly_stats = {
                'geometry_name': geometry_name,
                'year': year,
                'mean_lst_c': None,
                'std_lst_c': None,
                'median_lst_c': None,
                'q25_lst_c': None,
                'q75_lst_c': None,
            }
        else:
            # Compute statistics
            mean_val = (combined_yearly_collection.mean()
                        .reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=MODIS_SCALE, maxPixels=1e13)
                        .get('LST_Day_1km')
                        .getInfo())

            std_val = (combined_yearly_collection.reduce(ee.Reducer.stdDev())
                       .reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=MODIS_SCALE, maxPixels=1e13)
                       .get('LST_Day_1km_stdDev')
                       .getInfo())

            median_val = (combined_yearly_collection.reduce(ee.Reducer.percentile([50]))
                          .reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=MODIS_SCALE, maxPixels=1e13)
                          .get('LST_Day_1km_p50')
                          .getInfo())

            q25_val = (combined_yearly_collection.reduce(ee.Reducer.percentile([25]))
                       .reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=MODIS_SCALE, maxPixels=1e13)
                       .get('LST_Day_1km_p25')
                       .getInfo())

            q75_val = (combined_yearly_collection.reduce(ee.Reducer.percentile([75]))
                       .reduceRegion(reducer=ee.Reducer.mean(), geometry=geom, scale=MODIS_SCALE, maxPixels=1e13)
                       .get('LST_Day_1km_p75')
                       .getInfo())

            # Convert to Celsius
            mean_lst_c = mean_val * modis_factor - 273.15 if mean_val is not None else None
            std_lst_c = std_val * modis_factor if std_val is not None else None
            median_lst_c = median_val * modis_factor - 273.15 if median_val is not None else None
            q25_lst_c = q25_val * modis_factor - 273.15 if q25_val is not None else None
            q75_lst_c = q75_val * modis_factor - 273.15 if q75_val is not None else None

            yearly_stats = {
                'geometry_name': geometry_name,
                'year': year,
                'mean_lst_c': mean_lst_c,
                'std_lst_c': std_lst_c,
                'median_lst_c': median_lst_c,
                'q25_lst_c': q25_lst_c,
                'q75_lst_c': q75_lst_c,
            }
        all_yearly_stats.append(yearly_stats)

print("Finished processing all geometries and years. Total entries:", len(all_yearly_stats))


Processing geometry: gauntlet_northface


KeyboardInterrupt: 

**Reasoning**:
The previous code block executed successfully, indicating that all yearly statistics have been collected. To verify the data and prepare it for further analysis, I will now convert the `all_yearly_stats` list into a pandas DataFrame and display its head.



In [60]:
df_all_yearly_stats = pd.DataFrame(all_yearly_stats)
print(df_all_yearly_stat)


NameError: name 'df_all_yearly_stat' is not defined

## Create and Store DataFrames

### Subtask:
Assemble the calculated yearly statistics into a pandas DataFrame for each geometry, with years as the index and statistics as columns. Store these DataFrames in a dictionary.


**Reasoning**:
To organize the yearly statistics by geometry, I will iterate through the unique geometry names in 'df_all_yearly_stats', filter the data for each, set the 'year' column as the index, and store the resulting DataFrames in a dictionary.



In [50]:
geometry_dataframes = {}

for geometry_name in df_all_yearly_stats['geometry_name'].unique():
    # Filter data for the current geometry
    geometry_df = df_all_yearly_stats[df_all_yearly_stats['geometry_name'] == geometry_name].copy()

    # Set 'year' as index and remove 'geometry_name' and 'year' columns
    geometry_df.set_index('year', inplace=True)
    geometry_df.drop(columns=['geometry_name'], inplace=True)

    # Store the DataFrame in the dictionary
    geometry_dataframes[geometry_name] = geometry_df

# Print the keys of the dictionary to confirm the created DataFrames
print("DataFrames created for geometries:", geometry_dataframes.keys())


DataFrames created for geometries: dict_keys(['gauntlet_northface'])


## Display DataFrames

### Subtask:
Print each generated DataFrame, identifying it by its corresponding geometry.


**Reasoning**:
I need to iterate through the `geometry_dataframes` dictionary and print each DataFrame along with its corresponding geometry name as a header to fulfill the subtask.



In [51]:
for geometry_name, df in geometry_dataframes.items():
    print(f"\n--- DataFrame for {geometry_name} ---")
    print(df)


--- DataFrame for gauntlet_northface ---
      mean_lst_c  std_lst_c  median_lst_c  q25_lst_c  q75_lst_c
year                                                           
2002   -9.768549  15.524625    -12.261844 -21.808183   4.501879
2003  -13.863054  15.658020    -18.000838 -25.665825  -4.945963
2004  -14.328665  16.190711    -19.571586 -26.881208   1.769231
2005  -13.402676  15.016766    -12.173034 -25.866363  -1.431592
2006  -13.648049  15.300857    -16.913105 -24.836264  -2.234701
2007  -14.608611  14.747374    -18.182151 -25.503201  -5.987423
2008  -12.545085  18.084901    -16.323605 -28.801943   2.109577
2009  -11.166393  15.515998    -15.390532 -22.690973   0.301625
2010   -7.834806  14.231070    -10.697021 -18.737026   6.041339
2011  -12.657508  16.059883    -16.009161 -25.424414  -0.417222
2012  -14.154261  16.842324    -18.152534 -27.926999  -0.773923
2013  -13.021006  15.329872    -14.599637 -26.581792  -2.607741
2014  -11.210949  15.741839    -15.783408 -24.301510   3.61068

In [52]:
geometry_dataframes.items()


dict_items([('gauntlet_northface',       mean_lst_c  std_lst_c  median_lst_c  q25_lst_c  q75_lst_c
year                                                           
2002   -9.768549  15.524625    -12.261844 -21.808183   4.501879
2003  -13.863054  15.658020    -18.000838 -25.665825  -4.945963
2004  -14.328665  16.190711    -19.571586 -26.881208   1.769231
2005  -13.402676  15.016766    -12.173034 -25.866363  -1.431592
2006  -13.648049  15.300857    -16.913105 -24.836264  -2.234701
2007  -14.608611  14.747374    -18.182151 -25.503201  -5.987423
2008  -12.545085  18.084901    -16.323605 -28.801943   2.109577
2009  -11.166393  15.515998    -15.390532 -22.690973   0.301625
2010   -7.834806  14.231070    -10.697021 -18.737026   6.041339
2011  -12.657508  16.059883    -16.009161 -25.424414  -0.417222
2012  -14.154261  16.842324    -18.152534 -27.926999  -0.773923
2013  -13.021006  15.329872    -14.599637 -26.581792  -2.607741
2014  -11.210949  15.741839    -15.783408 -24.301510   3.610689
2015 

In [53]:
for geometry_name, df in geometry_dataframes.items():
  df.to_csv(f"4_{geometry_name}_2003-2025_yearly.csv")


## Mothly processing

In [68]:
monthly_lst_data = []

start_year = 2002
end_year = 2025
modis_factor = 0.02
MODIS_SCALE = 1000

for i, geom in enumerate(geoms):
    geometry_name = geom.getInfo().get("properties", {}).get("name", "unnamed")
    print(f'Processing: {geom_names[i]}')

    for year in range(start_year, end_year + 1):
        print(f"Processing year: {year}")

        for month in range(1, 13):

            start_date = ee.Date.fromYMD(year, month, 1)
            end_date = start_date.advance(1, 'month')

            monthly_coll = (
                modis
                .filterDate(start_date, end_date)
                .filterBounds(geom)
                .select("LST_Day_1km")
            )

            if monthly_coll.size().getInfo() == 0:
                continue

            # === REDUCTIONS ===
            mean_val = (
                monthly_coll.mean()
                .reduceRegion(
                    ee.Reducer.mean(),
                    geometry = geom,
                    scale = MODIS_SCALE,
                    maxPixels = 1e13
                )
                .get("LST_Day_1km")
                .getInfo()
            )

            std_val = (
                monthly_coll.reduce(ee.Reducer.stdDev())
                .reduceRegion(
                    ee.Reducer.mean(),
                    geometry = geom,
                    scale = MODIS_SCALE,
                    maxPixels = 1e13
                )
                .get("LST_Day_1km_stdDev")
                .getInfo()
            )

            q25_val = (
                monthly_coll.reduce(ee.Reducer.percentile([25]))
                .reduceRegion(
                    ee.Reducer.mean(),
                    geometry = geom,
                    scale = MODIS_SCALE,
                    maxPixels = 1e13
                )
                .get("LST_Day_1km_p25")
                .getInfo()
            )

            q75_val = (
                monthly_coll.reduce(ee.Reducer.percentile([75]))
                .reduceRegion(
                    ee.Reducer.mean(),
                    geometry = geom,
                    scale = MODIS_SCALE,
                    maxPixels = 1e13
                )
                .get("LST_Day_1km_p75")
                .getInfo()
            )

            # === APPLY SCALE + CONVERSION ===
            mean_c = mean_val * modis_factor - 273.15
            std_c  = std_val * modis_factor
            q25_c  = q25_val * modis_factor - 273.15
            q75_c  = q75_val * modis_factor - 273.15

            # === SAVE RECORD ===
            monthly_stats = {
                "geometry": geometry_name,
                "year": year,
                "month": month,
                "mean_c": mean_c,
                "std_c": std_c,
                "q25_c": q25_c,
                "q75_c": q75_c
            }

            monthly_lst_data.append(monthly_stats)

Processing: aulattiviup_fiordshore
Processing year: 2002
Processing year: 2003
Processing year: 2004
Processing year: 2005
Processing year: 2006
Processing year: 2007
Processing year: 2008
Processing year: 2009
Processing year: 2010
Processing year: 2011
Processing year: 2012
Processing year: 2013
Processing year: 2014
Processing year: 2015
Processing year: 2016
Processing year: 2017
Processing year: 2018
Processing year: 2019
Processing year: 2020
Processing year: 2021
Processing year: 2022
Processing year: 2023
Processing year: 2024
Processing year: 2025
Processing: ungujalik_cape
Processing year: 2002
Processing year: 2003
Processing year: 2004
Processing year: 2005
Processing year: 2006
Processing year: 2007
Processing year: 2008
Processing year: 2009
Processing year: 2010
Processing year: 2011
Processing year: 2012
Processing year: 2013
Processing year: 2014
Processing year: 2015
Processing year: 2016
Processing year: 2017
Processing year: 2018
Processing year: 2019
Processing yea

Processing year: 2025
Processing: kolik_valley
Processing year: 2002
Processing year: 2003
Processing year: 2004
Processing year: 2005
Processing year: 2006
Processing year: 2007
Processing year: 2008
Processing year: 2009
Processing year: 2010
Processing year: 2011
Processing year: 2012
Processing year: 2013
Processing year: 2014
Processing year: 2015
Processing year: 2016
Processing year: 2017
Processing year: 2018
Processing year: 2019
Processing year: 2020
Processing year: 2021
Processing year: 2022
Processing year: 2023
Processing year: 2024
Processing year: 2025
Processing: ujarasujjualuk_plateau
Processing year: 2002
Processing year: 2003
Processing year: 2004
Processing year: 2005
Processing year: 2006
Processing year: 2007
Processing year: 2008
Processing year: 2009
Processing year: 2010
Processing year: 2011
Processing year: 2012
Processing year: 2013
Processing year: 2014
Processing year: 2015
Processing year: 2016
Processing year: 2017
Processing year: 2018
Processing year:

In [85]:
df = pd.DataFrame(monthly_lst_data)


years = df["year"].nunique()
rows_per_geom = 12 * 24

# assign geometry names by slicing
df = df.sort_values(["geometry", "year", "month"]).reset_index(drop=True)

for i, geom in enumerate(geom_names):
    start = i * rows_per_geom
    end = start + rows_per_geom
    df.loc[start:end, "geometry"] = geom

In [86]:
df

,geometry,year,month,mean_c,std_c,q25_c,q75_c
0,aulattiviup_fiordshore,2002,1,-30.136717,7.967412,-35.894537,-25.429659
1,aulattiviup_fiordshore,2002,1,-31.253825,5.799561,-35.106206,-29.367343
2,aulattiviup_fiordshore,2002,1,-31.135229,5.367667,-34.992310,-28.896553
3,aulattiviup_fiordshore,2002,1,-31.282578,5.720817,-36.193891,-27.626484
4,aulattiviup_fiordshore,2002,1,-29.931720,6.330705,-34.723812,-26.841337
...,...,...,...,...,...,...,...
1651,tumbling_glacier,2024,12,-24.604070,5.233378,-28.602948,-20.147834
1652,tumbling_glacier,2024,12,-26.965448,5.713488,-31.471916,-22.061668
1653,tumbling_glacier,2024,12,-26.586719,6.283498,-30.586364,-21.577667
1654,tumbling_glacier,2024,12,-23.178782,4.423161,-27.131658,-19.770025


In [87]:
for i, geom in enumerate(geoms):
    gdf = df[df["geometry"] == geom_names[i]]
    if gdf.empty:
        print(f"Geometry missing: {geom}")
        continue

    # group by month and average the stats
    grouped = (
        gdf.groupby("month")[["mean_c", "std_c", "q25_c", "q75_c"]]
        .mean()
        .reset_index()
    )

    # export
    out_name = f"{geom_names[i]}_monthly.csv"
    grouped.to_csv(out_name, index=False)
    print("wrote:", out_name)

wrote: aulattiviup_fiordshore_monthly.csv
wrote: ungujalik_cape_monthly.csv
wrote: kolik_valley_monthly.csv
wrote: ujarasujjualuk_plateau_monthly.csv
wrote: gauntlet_northface_monthly.csv
wrote: tumbling_glacier_monthly.csv
